# Validate generated interpretations

Apply the original forensic validator to the common 35 events from DeepSeek, base Phi-4-mini and fine tuned Phi-4-mini.


## Input and output paths


In [ ]:
from pathlib import Path
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'pyproject.toml').is_file() and (p / 'data/forensic_atomic.csv').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Run this notebook from inside the forensic_atomic repository.')
import sys
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
DEEPSEEK_CSV = ROOT / 'results/phi4_mini/generation/raw/deepseek_generated_events.csv'
FINETUNED_JSONL = ROOT / 'results/phi4_mini/generation/raw/finetuned_selected_events_atomic_greedy_v1_max2048.jsonl'
BASE_JSONL = ROOT / 'results/phi4_mini/generation/raw/base_selected_events_atomic_greedy_v1_max2048.jsonl'
OUT = ROOT / 'results/phi4_mini/generation/validated'



ALLOW_FIRST_LIST_RECOVERY = True


ONLY_COMMON_EVENTS = True


## Load model responses


In [ ]:
import csv, json, re, logging
from collections import Counter, defaultdict
from IPython.display import display

RELATIONS = ['xIntent','xNeed','xAttr','xEffect','xReact','xWant','oEffect','oReact','oWant',
             'isAfter','HasSubEvent','isBefore','HinderedBy','Causes','xReason','isFilledBy']
FIELDS = ['event','crime_category','crime_subcategory','brief_context'] + RELATIONS

def find_file(name):
    p = Path(name).expanduser()
    if p.is_file(): return p.resolve()
    if (ROOT/p).is_file(): return (ROOT/p).resolve()
    found=sorted({x.resolve() for x in ROOT.rglob(p.name) if x.is_file()})
    if len(found)==1:return found[0]
    if len(found)>1:raise ValueError(f'Multiple files found for {name}: {found}; use the full path.')
    raise FileNotFoundError(f'Missing file: {name}')

def csv_read(path):
    with path.open(encoding='utf-8-sig',newline='') as f:
        return list(csv.DictReader(f))

def csv_write(path,rows,columns):
    path.parent.mkdir(parents=True,exist_ok=True)
    with path.open('w',encoding='utf-8-sig',newline='') as f:
        w=csv.DictWriter(f,fieldnames=columns,extrasaction='ignore');w.writeheader();w.writerows(rows)
    print('Written:',path,'| rows:',len(rows))

def jsonl_read(path):
    with path.open(encoding='utf-8') as f:
        return [json.loads(x) for x in f if x.strip()]

def norm_event(s):
    return ' '.join(str(s or '').casefold().split())

def model_event(r):
    return str((r.get('input_row') or {}).get('event') or r.get('event') or '').strip()

def relation_cell(v):
    if isinstance(v,list):return v
    if v is None:return None
    if isinstance(v,str):
        try: return json.loads(v)
        except json.JSONDecodeError: return v
    return v

def csv_entry(row):
    return {k:relation_cell(row.get(k)) if k in RELATIONS else row.get(k) for k in FIELDS if k!='event'}

def strip_generator_wrappers(raw):
    s=re.sub(r'<think>.*?</think>','',str(raw or ''),flags=re.DOTALL)
    s=s.replace('```json','').replace('```','').strip()
    return s

def first_full_json_list(text):
    decoder=json.JSONDecoder()
    for i,ch in enumerate(text):
        if ch=='[':
            try:
                obj,end=decoder.raw_decode(text[i:])
                if isinstance(obj,list): return obj,i,end
            except (json.JSONDecodeError,ValueError): pass
    return None,None,None

def parse_phi_response(rec):
    raw=rec.get('raw_response','')
    cleaned=strip_generator_wrappers(raw)
    try:
        obj=json.loads(cleaned)
        if isinstance(obj,list): return obj,'whole_output',[]
        return [],'non_list_root',[f'Radice JSON {type(obj).__name__}, attesa lista']
    except (json.JSONDecodeError,ValueError) as e:
        if not ALLOW_FIRST_LIST_RECOVERY:
            return [],'no_recovery',[str(e)]
        obj,start,end=first_full_json_list(cleaned)
        if obj is None: return [],'unrecoverable',[str(e)]
        return obj,'first_list_recovered',[f'Additional content after the first list; start={start}; end={start+end}']


## Forensic validator


In [ ]:
import json
from typing import List, Optional, Any
from pydantic import BaseModel, Field, field_validator
import logging
log = logging.getLogger("fatomic_original_validator_export")


class AtomicEntry(BaseModel):
    event: str
    crime_category: str = Field(..., description="Primary category")
    crime_subcategory: str = Field(..., description="Specific subcategory")
    brief_context: Optional[str] = Field(None, description="Context needed for this interpretation")


    xIntent: List[str] = Field(default_factory=list)
    xNeed: List[str] = Field(default_factory=list)
    xAttr: List[str] = Field(default_factory=list)
    xEffect: List[str] = Field(default_factory=list)
    xReact: List[str] = Field(default_factory=list)
    xWant: List[str] = Field(default_factory=list)
    oEffect: List[str] = Field(default_factory=list)
    oReact: List[str] = Field(default_factory=list)
    oWant: List[str] = Field(default_factory=list)


    isAfter: List[str] = Field(default_factory=list)
    HasSubEvent: List[str] = Field(default_factory=list)
    isBefore: List[str] = Field(default_factory=list)
    HinderedBy: List[str] = Field(default_factory=list)
    Causes: List[str] = Field(default_factory=list)
    xReason: List[str] = Field(default_factory=list)
    isFilledBy: List[str] = Field(default_factory=list)

    @field_validator('crime_category')
    @classmethod
    def standardize_category(cls, v):
        """Standardize the Non-Criminal category"""
        v_lower = v.lower().strip()
        if v_lower in ["no crime", "not a crime", "innocent", "legal", "none"]:
            return "Non-Criminal"
        return v.strip()


    @field_validator(
        'xIntent', 'xWant', 'oReact', 'xReact', 'oWant', 'xEffect', 'oEffect', 'xNeed', 'xAttr',
        'isAfter', 'HasSubEvent', 'isBefore', 'HinderedBy', 'Causes', 'xReason', 'isFilledBy',
        mode='before'
    )
    @classmethod
    def force_list(cls, v: Any):
        """
        If the LLM returns an empty string or 'none' instead of a list,
        convert it to an empty list to prevent Pydantic crash.
        """
        if isinstance(v, str):
            v_clean = v.strip().lower()
            if v_clean == "" or v_clean in ["none", "null", "n/a"]:
                return []

            return [v]
        if v is None:
            return []
        return v

    @field_validator(
        'xIntent', 'xWant', 'oReact', 'xReact', 'oWant', 'xEffect', 'oEffect', 'xNeed', 'xAttr',
        'isAfter', 'HasSubEvent', 'isBefore', 'HinderedBy', 'Causes', 'xReason', 'isFilledBy'
    )
    @classmethod
    def check_content_quality(cls, v, info):
        """
        Validates that list items are not empty, not too short, and do not contain AI filler words.
        """
        if not v:
            return v


        cleaned = [item.strip() for item in v if len(item.strip()) >= 3]


        forbidden = [
            "i don't know", "depends on context", "as an ai", "unknown",
            "n/a", "not applicable", "i cannot", "ai model", "sorry", "illegal"
        ]

        final_list = []
        for item in cleaned:
            if not any(bad in item.lower() for bad in forbidden):
                final_list.append(item)

        return final_list


def validate_llm_output(json_data: Any, original_event: str) -> Optional[dict]:
    """
    Validates the raw dictionary from the LLM against the AtomicEntry schema.
    Returns the validated dictionary or None if validation fails.
    """


    if isinstance(json_data, str):
        if "not possible" in json_data.lower():
            log.warning(f"Impossible to create a criminal context for '{original_event}'")
            return None
        else:
            log.warning(f"Validation failed for '{original_event}': Expected dictionary, got string.")
            return None
    elif isinstance(json_data, list) and len(json_data) > 0 and isinstance(json_data[0], str) and "not possible" in json_data[0].lower():
        log.warning(f"Impossible to create a criminal context for '{original_event}'")
        return None


    if isinstance(json_data, list) and len(json_data) > 0 and isinstance(json_data[0], str):
        if "not possible" in json_data[0].lower():
            log.warning(f"Impossible to create a criminal context for '{original_event}'")
            return None


    if not isinstance(json_data, dict):
        log.warning(f"Validation failed for '{original_event}': Expected dictionary, got {type(json_data).__name__}")
        return None

    try:

        data_to_validate = json_data.copy()
        data_to_validate['event'] = original_event


        entry = AtomicEntry(**data_to_validate)
        return entry.model_dump()
    except Exception as e:
        log.warning(f"Validation failed for '{original_event}': {e}")
        return None


## Validate common events


In [ ]:
paths={ 'deepseek':find_file(DEEPSEEK_CSV),
        'finetuned':find_file(FINETUNED_JSONL),
        'base':find_file(BASE_JSONL) }
source_data={m: (csv_read(p) if m=='deepseek' else jsonl_read(p)) for m,p in paths.items()}
by_model={}
for model,rows in source_data.items():
    g=defaultdict(list)
    for row in rows:
        ev=str(row.get('event','')).strip() if model=='deepseek' else model_event(row)
        if not ev: raise ValueError(f'Empty event in {model}')
        g[norm_event(ev)].append(row)
    if model!='deepseek' and any(len(v)!=1 for v in g.values()):
        raise ValueError(f'Duplicate events in JSONL {model}: check the sources.')
    by_model[model]=g
keys=set.intersection(*(set(v) for v in by_model.values())) if ONLY_COMMON_EVENTS else set.union(*(set(v) for v in by_model.values()))
print('Common events:',len(keys),'| by model:',{m:len(g) for m,g in by_model.items()})
if not keys: raise ValueError('No common events')

exports={m:[] for m in by_model}; branch_trace=[];event_trace=[]; rejected=[]
for key in sorted(keys):

    event= (model_event(by_model['finetuned'][key][0]) if key in by_model['finetuned']
            else (by_model['deepseek'][key][0]['event'] if key in by_model['deepseek'] else key))
    for model in ('deepseek','finetuned','base'):
        src=by_model[model].get(key,[])
        if model=='deepseek':
            entries=[csv_entry(x) for x in src]; method='csv_gia_elaborato';orig_status='csv_no_raw'
        elif src:
            entries,method,_=parse_phi_response(src[0]);orig_status=src[0].get('status','')
        else: entries=[];method='missing_event';orig_status='missing_event'
        n_ok=0
        for j,entry in enumerate(entries,1):
            normalized=validate_llm_output(entry,event)
            accepted=normalized is not None
            changed_fields=[k for k in FIELDS if k!='event' and isinstance(entry,dict) and normalized is not None and entry.get(k) != normalized.get(k)]
            branch_trace.append({'model':model,'event':event,'branch_index':j,'accepted_validator':int(accepted),
                                 'origin_method':method,'raw_status':orig_status,
                                 'category_raw':entry.get('crime_category','') if isinstance(entry,dict) else '',
                                 'subcategory_raw':entry.get('crime_subcategory','') if isinstance(entry,dict) else '',
                                 'isFilledBy_nonempty':int(bool(entry.get('isFilledBy'))) if isinstance(entry,dict) else '',
                                 'has_placeholder':int('___' in event),
                                 'campi_modificati_da_validator':';'.join(changed_fields)})
            if accepted:
                n_ok+=1
                serialized={k:(json.dumps(normalized[k],ensure_ascii=False) if k in RELATIONS
                                else ('' if normalized.get(k) is None else normalized.get(k,''))) for k in FIELDS}
                exports[model].append(serialized)
            else:
                rejected.append({'model':model,'event':event,'branch_index':j,
                                 'reason':'Rejected by the original validator',
                                 'raw_json':json.dumps(entry,ensure_ascii=False,default=str)})
        event_trace.append({'model':model,'event':event,'raw_status':orig_status,'extraction_method':method,
                            'raw_truncated':int(bool(src[0].get('hit_max_tokens'))) if model!='deepseek' and src else '',
                            'n_candidates':len(entries),'n_accepted_validator':n_ok,
                            'n_rejected_validator':len(entries)-n_ok})

OUT.mkdir(parents=True,exist_ok=True)
for model,rows in exports.items():
    csv_write(OUT/f'{model}_post_validator_pre_cleaning.csv',rows,FIELDS)
csv_write(OUT/'event_trace.csv',event_trace,
          ['model','event','raw_status','extraction_method','raw_truncated','n_candidates','n_accepted_validator','n_rejected_validator'])
csv_write(OUT/'interpretation_trace.csv',branch_trace,
          ['model','event','branch_index','accepted_validator','origin_method','raw_status','category_raw','subcategory_raw','isFilledBy_nonempty','has_placeholder','campi_modificati_da_validator'])
csv_write(OUT/'rejected_interpretations.csv',rejected,
          ['model','event','branch_index','reason','raw_json'])
print('Summary by model:')
for model in ('deepseek','finetuned','base'):
    e=[r for r in event_trace if r['model']==model]
    print(model, 'events:',len(e),'validated branches:',len(exports[model]),
          'covered events:',sum(x['n_accepted_validator']>0 for x in e),
          'first-list recoveries:',sum(x['extraction_method']=='first_list_recovered' for x in e),
          'normalized branches:',sum(bool(x['campi_modificati_da_validator']) for x in branch_trace if x['model']==model))


## Outputs
